### Summarize website content

In [ ]:
import os
from dotenv import load_dotenv

from openai import OpenAI

load_dotenv(override=True)

In [ ]:
openai = OpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"), api_key=os.getenv("OPENAI_API_KEY")
)

In [ ]:
from pydantic import BaseModel
from typing import Literal


class OpenAIMessage(BaseModel):
    role: Literal["system", "user"]
    content: str

In [ ]:
messages: list[OpenAIMessage] = [
    OpenAIMessage(role="system", content="Your are a helpful assistant"),
    OpenAIMessage(role="user", content="What is 2 + 2?"),
]

In [ ]:
response = openai.chat.completions.create(messages=messages, model="gemma3:1b")  # type: ignore
print(response.choices[0].message.content)

In [ ]:
SYSTEM_PROMPT = """
You are a helpful assistant that analyzes the contents of a website, and provides a short summary, 
ignoring text that might be navigation related. Response in markdown. Do not wrap the markdown in code block 
- respond just with the markdown
"""

USER_PROMPT_PREFIX = """
Here are the content of the website. Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

In [ ]:
from advanced_scraper import fetch_page_content


async def summarize_webpage(url: str) -> str:
    try:
        webpage = await fetch_page_content(url=url)
        messages = [
            OpenAIMessage(role="system", content=SYSTEM_PROMPT),
            OpenAIMessage(role="user", content=USER_PROMPT_PREFIX + webpage),
        ]
        response = openai.chat.completions.create(messages=messages, model="gemma3:1b")  # type:ignore
        summary = response.choices[0].message.content
        return summary or "No summary found"
    except Exception as ex:
        print(ex)
        return "Internal server error. Please re-try after sometime"

In [ ]:
from IPython.display import display, Markdown  # type: ignore


async def display_summary(url: str) -> None:
    summary = await summarize_webpage(url=url)
    display(Markdown(summary))

In [ ]:
await display_summary(url="https://edwarddonner.com")

In [ ]:
await display_summary(url="https://cnn.com")